# Benchmark de precios publicados de venta en CABA

Estimamos USD por m² cubierto con cinco familias y una referencia HGB fija. La unidad de evaluación es el aviso. La unidad de partición y remuestreo es la propiedad aproximada por `grupo`. El modelo usa únicamente las 12 variables base y, en el segundo escenario, antigüedad y estado.

Semilla 42, cinco folds externos, tres internos, tres configuraciones por familia, un hilo. Dos escenarios suman hasta 500 ajustes de búsqueda y refit, más cinco ajustes de la referencia. La búsqueda minimiza MAE en USD/m². La red tiene dos capas ocultas de 48 y 24 unidades.

El catálogo corresponde a la captura del 7 al 9 de noviembre de 2026. Los delitos registrados se muestran como contexto por barrio y año en una tabla separada.

In [ ]:
import hashlib
import sys
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'precio_entorno').is_dir())
sys.path.insert(0, str(root))
from precio_entorno.benchmark_inmobiliario import (
    BASELINE, BUNDLE, CATEGORICAL, EXTRA, NUMERIC, benchmark, bootstrap_mae_delta, load_sales, ranking, scores,
)

data = load_sales(BUNDLE)
print('scikit-learn', sklearn.__version__, '· SHA-256 del bundle', hashlib.sha256(BUNDLE.read_bytes()).hexdigest())
print(len(data), 'avisos ·', data.grupo.nunique(), 'grupos ·', data.neighborhood.nunique(), 'barrios')
print('Grupos repetidos:', data.groupby('grupo').size().gt(1).sum())
print('Grupos en múltiples fuentes:', data.groupby('grupo').agency.nunique().gt(1).sum())
print('Mayor diferencia de precio/m² contra precio / cubierta:', (data.price / data.covered_area - data.precio_m2).abs().max())
features = NUMERIC + CATEGORICAL + EXTRA
missing = data[features].isna().mul(100)
display(missing.mean().rename('faltantes_pct').round(1))
display(missing.groupby(data.agency).mean().round(1), missing.groupby(data.neighborhood).mean().round(1))
display(data[['precio_m2', 'covered_area', 'total_area_m2', 'latitude', 'longitude']].describe(percentiles=[.01, .25, .5, .75, .99]))
print('Cubierta mayor al total:', data.covered_area.gt(data.total_area_m2).sum())
print('Pines repetidos después del primero:', data.dropna(subset=['latitude', 'longitude']).duplicated(['latitude', 'longitude']).sum())
with ZipFile(BUNDLE) as source, source.open('delitos/barrio_anio.csv') as records:
    context = pd.read_csv(records)
display(context.groupby('anio').agg(barrios=('barrio', 'nunique'), hechos=('total_delitos', 'sum')))

In [ ]:
target = data.precio_m2
relative = target / data.groupby('neighborhood').precio_m2.transform('median')
neighborhoods = data.groupby('neighborhood')[['precio_m2', *NUMERIC, 'age_years']].median()
associations = []
for feature in NUMERIC + ['age_years']:
    x = data[feature]
    pairs = neighborhoods[[feature, 'precio_m2']].dropna()
    associations.append({'variable': feature, 'n_avisos': x.notna().sum(), 'Pearson': x.corr(target),
                         'Spearman': x.corr(target, method='spearman'),
                         'Spearman_relativo': x.corr(relative, method='spearman'),
                         'n_barrios': len(pairs), 'Spearman_mediana_barrio': pairs[feature].corr(pairs.precio_m2, method='spearman')})
display(pd.DataFrame(associations).round(3))
display(data.dropna(subset=['age_years']).groupby(pd.cut(data.age_years, [-1, 0, 10, 30, 50, 70, np.inf]), observed=True)
        .precio_m2.agg(n='size', p25=lambda s: s.quantile(.25), mediana='median', p75=lambda s: s.quantile(.75)))

## Evaluación fuera de muestra

La transformación inversa es exp(predicción log). La media condicional en escala USD puede diferir por la convexidad de la exponencial. La brecha comercial usa la predicción como denominador. MdAPE usa el precio observado. Las dos especificaciones entrenan sobre las mismas filas, y la evaluación con antigüedad conocida usa el mismo subconjunto de prueba para ambas.

In [ ]:
oof, fits, interpretation = benchmark(data)
assert oof.groupby(['model', 'scope']).row.nunique().eq(len(data)).all()
assert oof.groupby('grupo').fold.nunique().eq(1).all()
assert not oof.duplicated(['model', 'scope', 'row']).any()
summary = []
for sample, rows in [('total', oof), ('edad informada', oof[oof.age_years.notna()])]:
    baseline = rows[rows.model == BASELINE]
    for (model, scope), block in rows.groupby(['model', 'scope']):
        low, high = bootstrap_mae_delta(block, baseline)
        fold_mae = block.error.abs().groupby(block.fold).mean()
        summary.append({'sample': sample, 'model': model, 'scope': scope, **scores(block),
                        'delta_mae_vs_eda': scores(block).mae - scores(baseline).mae,
                        'delta_low': low, 'delta_high': high, 'fold_mae_sd': fold_mae.std()})
summary = pd.DataFrame(summary).sort_values(['sample', 'mae'])
display(summary.round(2))
display(fits.groupby(['model', 'scope']).seconds.agg(['mean', 'min', 'max', 'sum']).round(2))
display(fits[['fold', 'model', 'scope', 'params']])

In [ ]:
rank = ranking(oof, k=20)
display(rank.groupby(['model', 'scope', 'threshold']).agg(precision_proxy=('precision_proxy', 'mean'),
        precision_sd=('precision_proxy', 'std'), jaccard=('jaccard_vs_eda', 'mean'), coverage=('coverage', 'mean'),
        n_top=('top_n', 'sum'), flags=('flags_in_top_without_review', 'sum'), candidates=('candidates', 'sum')).round(3))
for column in ['agency', 'neighborhood']:
    profile = [dict(zip(['model', 'scope', column], key)) | scores(block).to_dict()
               for key, block in oof.groupby(['model', 'scope', column])]
    display(pd.DataFrame(profile).round(2))

In [ ]:
winner = summary[(summary['sample'] == 'total') & summary.model.isin(['Random Forest', 'HistGradientBoosting'])].iloc[0]
chosen = interpretation[(interpretation.model == winner.model) & (interpretation.scope == winner.scope)]
coef = interpretation[interpretation.model == 'Ridge'].groupby(['scope', 'feature']).value.agg(['mean', 'std'])
coef['cambio_pct_por_unidad'] = 100 * np.expm1(coef['mean'])
display(coef.round(4))
display(chosen[chosen.type == 'aumento_mae_usd_m2'].groupby('feature').value.agg(['mean', 'std']).sort_values('mean', ascending=False))
pdp = chosen[chosen.type == 'pdp_usd_m2'].groupby('quantile')[['area', 'value', 'ice_p10', 'ice_p90']].mean()
display(pdp.round(1))
pdp.plot(x='area', y=['value', 'ice_p10', 'ice_p90'], ylabel='USD/m² publicados', xlabel='Superficie cubierta (m²)', title=f'{winner.model} · {winner.scope}');

La precisión del ranking mide coincidencia con un criterio de comparables construido en entrenamiento. Hace falta revisión de avisos y seguimiento de capturas sucesivas para validar su utilidad en alertas. La importancia por permutación depende de la correlación entre variables. Las curvas PDP/ICE pueden combinar superficies con ambientes o superficies totales poco plausibles.